# 03 ? Retrieval Evaluation

**Objective:** Evaluate the existing Qdrant retrieval system on the project's golden dataset.

The dataset currently has 25 questions, but only questions with manually verified `relevant_chunk_ids` are included in ranking metrics. The verified subset is reported explicitly.

This notebook compares two abstention thresholds (0.20 and 0.30) using the same retrieved results. It does not claim that changing the threshold changes ranking metrics.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import sys
import json
from pathlib import Path
import pandas as pd

# Find project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "app").exists():
    raise FileNotFoundError("Project root not found. Check notebook location.")

# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

from app.config import DEFAULT_TOP_K
from app.ingestion.vector_store import search_chunks
from app.evaluation.metrics import (
    hit_rate_at_k,
    recall_at_k,
    reciprocal_rank,
    mean_reciprocal_rank
)

# Load golden dataset
dataset_path = PROJECT_ROOT / "app" / "evaluation" / "golden_dataset.json"

with dataset_path.open("r", encoding="utf-8") as file:
    dataset = json.load(file)

print("Project root:", PROJECT_ROOT)
print("Golden questions:", len(dataset))
print("Dataset loaded successfully.")

Project root: C:\portfolio 1\IEC-Production-RAG--Services
Golden questions: 25
Dataset loaded successfully.


## 1. Retrieve results once for each question

The vector store must already contain the corpus used to create the golden dataset. If Qdrant is empty or uses a different corpus, stop and ingest the matching documents first.

In [5]:
retrieval_cache = {}

for item in dataset:
    matches = search_chunks(item["question"], top_k=DEFAULT_TOP_K)

    retrieved_ids = [
        f'{match["metadata"]["document_id"]}_{match["metadata"]["position"]}'
        for match in matches
    ]

    retrieval_cache[item["id"]] = {
        "matches": matches,
        "retrieved_ids": retrieved_ids,
        "max_similarity": max(
            (match["similarity"] for match in matches),
            default=0.0,
        ),
    }

print("Questions retrieved:", len(retrieval_cache))

Questions retrieved: 25


## 2. Compare abstention thresholds

A question is marked as abstained when there are no matches or the highest similarity is below the selected threshold.

In [6]:
threshold_rows = []
question_rows = []

for threshold in [0.20, 0.30]:
    abstained_answerable = 0
    accepted_unanswerable = 0
    abstained_total = 0

    for item in dataset:
        cached = retrieval_cache[item["id"]]
        abstained = (
            not cached["matches"]
            or cached["max_similarity"] < threshold
        )

        if abstained:
            abstained_total += 1
            if item["answerable"]:
                abstained_answerable += 1
        elif not item["answerable"]:
            accepted_unanswerable += 1

        question_rows.append({
            "threshold": threshold,
            "question_id": item["id"],
            "answerable": item["answerable"],
            "max_similarity": round(cached["max_similarity"], 4),
            "abstained": abstained,
        })

    threshold_rows.append({
        "threshold": threshold,
        "total_questions": len(dataset),
        "abstained_total": abstained_total,
        "answerable_questions_abstained": abstained_answerable,
        "unanswerable_questions_not_abstained": accepted_unanswerable,
    })

threshold_comparison = pd.DataFrame(threshold_rows)
threshold_comparison

,threshold,total_questions,abstained_total,answerable_questions_abstained,unanswerable_questions_not_abstained
0,0.2,25,2,0,3
1,0.3,25,6,1,0


## 3. Ranking metrics on the verified subset

Hit Rate@5, Recall@5 and MRR are computed only for questions with manually verified relevant chunk IDs. Unlabelled questions are not treated as having zero relevant results.

In [7]:
verified_rows = []

for item in dataset:
    relevant_ids = item.get("relevant_chunk_ids", [])
    if not relevant_ids:
        continue

    retrieved_ids = retrieval_cache[item["id"]]["retrieved_ids"]

    verified_rows.append({
        "question_id": item["id"],
        "hit_rate_at_5": hit_rate_at_k(retrieved_ids, relevant_ids, 5),
        "recall_at_5": recall_at_k(retrieved_ids, relevant_ids, 5),
        "reciprocal_rank": reciprocal_rank(retrieved_ids, relevant_ids),
    })

verified_df = pd.DataFrame(verified_rows)
display(verified_df)

if verified_rows:
    ranking_summary = pd.DataFrame([{
        "verified_question_count": len(verified_rows),
        "hit_rate_at_5": verified_df["hit_rate_at_5"].mean(),
        "recall_at_5": verified_df["recall_at_5"].mean(),
        "mrr": mean_reciprocal_rank([
            (
                retrieval_cache[row["question_id"]]["retrieved_ids"],
                next(
                    item["relevant_chunk_ids"]
                    for item in dataset
                    if item["id"] == row["question_id"]
                ),
            )
            for row in verified_rows
        ]),
    }])
    display(ranking_summary)
else:
    print("No manually verified relevant chunk IDs are available.")

,question_id,hit_rate_at_5,recall_at_5,reciprocal_rank
0,1,1.0,1.0,1.000000
1,2,1.0,1.0,1.000000
2,3,1.0,1.0,0.500000
3,4,1.0,1.0,0.500000
4,5,1.0,1.0,0.333333


,verified_question_count,hit_rate_at_5,recall_at_5,mrr
0,5,1.0,1.0,0.666667


## 4. Evaluation limitations and next steps

1. Only manually labelled questions contribute to ranking metrics. Expand the verified labels before making broad quality claims.
2. Threshold comparison measures abstention trade-offs, not embedding or chunking quality.
3. A proper chunking comparison requires separate vector-store collections populated with each configuration and evaluated against the same gold labels.
4. Retrieval metrics do not prove answer correctness or faithfulness.
5. Preserve the corpus, model name, chunk settings, top-K and threshold when reporting reproducible results.